<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

در این کد:
1. پاسخ را به ادعاهای مستقل تقسیم می‌کنیم.
2. هر ادعا را با Context مقایسه می‌کنیم.
3. مشخص می‌کنیم آیا ادعا پشتیبانی می‌شود یا خیر.
4. امتیاز Faithfulness را محاسبه می‌کنیم.

In [ ]:
import json
from openai import OpenAI

# 1. Connect to Ollama
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

MODEL_ID = "qwen3:4b"

# 2. Retrieved contexts
retrieved_contexts = [
    "Employees receive 26 days of annual leave per year.",
    "All annual leave requests require manager approval.",
]

# 3. Generated answer
generated_answer = (
    "Employees receive 26 days of annual leave. "
    "All leave requests require manager approval. "
    "Employees can take 10 additional paid days."
)

# 4. Evaluation prompt
prompt = f"""
Check which claims in the answer are supported by the context.

Context:
{json.dumps(retrieved_contexts)}

Answer:
{generated_answer}

Split the answer into atomic claims.
For each claim, set supported to true or false.

Return JSON only:
{{
  "claims": [
    {{"claim": "Claim text", "supported": true}}
  ]
}}
"""

# 5. Ask Qwen3
response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
    extra_body={"think": False},
)

# 6. Parse result
result_text = response.choices[0].message.content

try:
    result = json.loads(result_text)
    claims = result["claims"]

    if not isinstance(claims, list) or not claims:
        raise ValueError("No claims returned.")

    supported_count = 0

    for item in claims:
        if not isinstance(item["claim"], str):
            raise ValueError("Invalid claim.")

        if not isinstance(item["supported"], bool):
            raise ValueError("Invalid support value.")

        if item["supported"]:
            supported_count += 1

    # 7. Calculate Faithfulness
    faithfulness = supported_count / len(claims)

    print("Claim Evaluations:")

    for item in claims:
        status = "SUPPORTED" if item["supported"] else "NOT SUPPORTED"
        print(f"\n{item['claim']}")
        print(status)

    print(f"\nFaithfulness: {faithfulness:.2f}")

except (json.JSONDecodeError, KeyError, TypeError, ValueError) as error:
    print("Evaluation failed:", error)
    print("Model response:", result_text)
